# Optimizing the Holevo information of an ensemble of reflections

Numerics for the section **"Optimization of the Holevo information"** of
*Quantum Programmable Reflections* (Schoute, Grinko, Subaşı, Volkoff).

For a uniform ensemble of reflections $R = e^{i\pi|\psi\rangle\langle\psi|}$ applied to an
$n$-copy entangled probe state $|\Phi_n\rangle$, the reflected ensemble is block diagonal
in the mixed Schur basis.  In block $\nu$ it is determined by a vector $|v_\nu\rangle$ whose
squared norm is a *linear* function of the probe-state symmetry weights $\{q_\lambda\}$,

$$\lVert v_\nu\rVert^2 \;=\; (K\,q)_\nu , \qquad
S\big(\widetilde{\mathcal T(\rho)}\big) \;=\; -\sum_\nu d_\nu\,\lVert v_\nu\rVert^2 \log_2 \lVert v_\nu\rVert^2 .$$

This notebook builds the transition matrix $K$ from $U(d)$ Clebsch–Gordan coefficients and
maximizes the entropy over $\{q_\lambda\}$, then compares the result to the upper bound
$2\log_2 D(n,d)$ with $D(n,d)=\binom{n+d-1}{d-1}^2$.  It also covers the qubit special case
and the comparison with the Yang–Renner–Chiribella (YRC) probe state.

In [ ]:
# Activate the pinned Julia environment (Project.toml), searching this directory and its parents.
import Pkg
function find_project(dir = pwd())
    while !isfile(joinpath(dir, "Project.toml"))
        parent = dirname(dir)
        parent == dir && error("Could not find Project.toml -- run this notebook from the repository directory.")
        dir = parent
    end
    return dir
end
Pkg.activate(find_project())
# Pkg.instantiate()   # uncomment on first run to install the pinned dependencies

In [ ]:
using SUNRepresentations          # SU(d) irreps, GT patterns, Clebsch–Gordan coefficients
using Combinatorics               # integer partitions -> SU(d) irrep labels
using LinearAlgebra
using Memoize                     # cache the (expensive) transition-matrix computations
using JuMP, Ipopt                 # entropy maximization over the weights q_λ
using Plots

## Gelfand–Tsetlin pattern utilities

A Gelfand–Tsetlin (GT) pattern `L` of shape $\lambda$ labels a basis vector of the Weyl module
$W_\lambda$ of $SU(d)$.  The helpers below implement the index bookkeeping and the sign/phase
factors $(-1)^{w_d(L)}$ and $\varphi(L)$ that appear when the reflection is written in the GT
basis (see the lemmas on the reflection and the conjugate GT basis in the paper).

In [ ]:
# Sum of the l-th row of a GT pattern (rows counted from the bottom; row 0 is empty).
rowsum(m, l) = l == 0 ? 0 : sum(m[k, l] for k in 1:l)

# w_d(L): the last entry of the weight of the GT pattern L.  The reflection acts on the GT
# basis vector |L> with the sign (-1)^{w_d(L)}.
wd(GTpat) = weight(GTpat)[end]

# λ = (λ_1, …, λ_d)  ->  the SU(d) label (λ_1-λ_d, …, λ_{d-1}-λ_d, 0) expected by SUNRepresentations.
SUi(λ) = Tuple(λ[i] - λ[end] for i in eachindex(λ))

# Pad a partition (given as a short vector) with trailing zeros to length `len`.
padright(arr, len) = length(arr) >= len ? arr : vcat(arr, fill(0, len - length(arr)))

# Position of a GT pattern in the canonical basis of its irrep.
function GTindex(GTpat)
    d = length(weight(GTpat))
    λ = Tuple(GTpat[i, d] for i in 1:d)
    return findfirst(==(GTpat), collect(basis(SUNIrrep(λ))))
end

# The GT pattern of the conjugate irrep obtained from L (used by Lemma "conj GT basis").
function dualGTpat(GTpat)
    d = length(weight(GTpat))
    out = Base.setindex(GTpat, GTpat[1, d] - GTpat[1, 1], 1, 1)
    for l in 2:d, k in 1:l
        out = Base.setindex(out, GTpat[1, d] - GTpat[k, l], l - k + 1, l)
    end
    return out
end

# The "maximal" GT pattern of the same shape as L (every interlacing entry pushed up).
function maxGTpat(GTpat)
    d = length(weight(GTpat))
    out = Base.setindex(GTpat, GTpat[1, d], 1, 1)
    for l in 2:d, k in 1:l
        out = Base.setindex(out, GTpat[k, d], k, l)
    end
    return out
end

# φ(L): the phase exponent in the conjugate-basis transformation, normalized so that φ(L_max) = 0.
φ₀(GTpat) = sum(rowsum(GTpat, i) for i in 1:(length(weight(GTpat)) - 1))
φ(GTpat)  = φ₀(GTpat) - φ₀(maxGTpat(GTpat))

## Irreps appearing in the construction

In [ ]:
# Partitions of n into exactly d parts (zero-padded), as length-d tuples.
all_partitions_d(n, d) = (n != 0 && d != 0) ? [Tuple(padright(λ, d)) for λ in partitions(n, d)] :
                                               [Tuple(zeros(Int, d))]

# Partitions of n into at most d parts (zero-padded), as length-d tuples.
all_partitions(n, d) = (n != 0 && d != 0) ? [Tuple(padright(λ, d)) for l in 1:d for λ in partitions(n, l)] :
                                             [Tuple(zeros(Int, d))]

# λ ⊢_d n : the SU(d) irreps that may carry weight in the probe state |Φ_n>.
λirreps(n, d) = all_partitions(n, d)

# Irreps ν of the partially transposed permutation algebra (mixed Schur–Weyl duality).
function mixed_irreps(n, d)
    labels = [padright(collect(λ), d) - reverse(padright(collect(μ), d))
              for k in 0:(n - 1) for l in 1:d-1
              for λ in all_partitions_d(n - k, l) for μ in all_partitions(n - k, d - l)]
    push!(labels, zeros(Int, d))
    return SUNIrrep.(SUi.(Tuple.(labels)))
end

# The single family ν_k = (k, 0, …, 0, -k) that supports the symmetric-subspace ensemble.
ν_irrep(k, d) = SUNIrrep(SUi(Tuple(vcat(k, fill(0, d - 2), -k))))

# D(n,d) = binomial(n+d-1, d-1)^2.  The Holevo information is upper bounded by 2 log2 D(n,d).
D(n, d) = binomial(Int128(n + d - 1), d - 1)^2

# Conjectured maximal Holevo information: log2 dim Sym^n(C^2) for d = 2, else 2 log2 d_θ.
entropy_guess(n, d) = d == 2 ? log2(binomial(Int128(n + 2), 2)) :
                               log2(binomial(Int128(n + d - 1), d - 1)^2)

# Alicki–Fannes–Winter-adjusted lower bound on log2 d_P for programming error ε.
lb_guess(m, ε, d) = entropy_guess(m, d) - 4 * m * sqrt(2 * ε) * log2(D(m, d)) - 1

# GT pattern of the highest weight of ν_k = (k, 0, …, 0, -k).
function NGT_pat(k, d)
    pat = collect(basis(mixed_irreps(k, d)[1]))[1]
    for j in 1:d, i in 1:j
        pat = Base.setindex(pat, 0, i, j)
    end
    pat = Base.setindex(pat, k,  1, d)
    pat = Base.setindex(pat, -k, d, d)
    return pat
end

## The transition matrix $K$

`LinSystem(n, d)` returns `(K, target)` where `K` maps a vector of probe weights $q_\lambda$
to the squared norms $\lVert v_\nu\rVert^2$, and `target[k] = d_{\nu_k} / D(n,d)` is the
distribution those squared norms would have to match to saturate the upper bound.
The qubit case is handled separately because there the spins, rather than partitions, label
the relevant data.

In [ ]:
@memoize function LinSystemQubits(n)
    # d = 2: the transition matrix and target distribution indexed by total spin J.
    K      = zeros(div(n, 2) + 1, div(n, 2) + 1)
    target = zeros(div(n, 2) + 1)
    for (J_index, J) in enumerate(n % 2 : 2 : n)
        r = floor(Int, n / 2 - J / 2 + 1)
        ν = SUNIrrep(2 * J, 0)
        N = collect(basis(ν))[J + 1]
        for i in 1:r
            j = J // 2 + ((n - J) % 2) // 2 + i - 1
            i_index = Int(j - (n % 2) // 2) + 1
            amp = sum(CGC(SUNIrrep(Int(2j), 0), conj(SUNIrrep(Int(2j), 0)), ν)[GTindex(L), GTindex(dualGTpat(L)), GTindex(N), 1]
                      for L in basis(SUNIrrep(Int(2j), 0)))
            K[J_index, i_index] = amp^2 / (2j + 1)
        end
        target[J_index] = (2J + 1) / binomial(n + 2, 2)
    end
    return K, target
end

@memoize function LinSystem(n, d)
    d == 2 && return LinSystemQubits(n)

    irreps = λirreps(n, d)
    K      = zeros(Float64, n + 1, length(irreps))
    target = zeros(Float64, n + 1)
    for k in 0:n
        ν = ν_irrep(k, d)
        N = NGT_pat(k, d)
        for (i, λ_label) in enumerate(irreps)
            λ = SUNIrrep(SUi(λ_label))
            haskey(directproduct(λ, conj(λ)), ν) || continue
            for t in 1:directproduct(λ, conj(λ))[ν]
                K[k + 1, i] += sum((-1)^(wd(L) + φ(L)) *
                                   CGC(λ, conj(λ), ν)[GTindex(L), GTindex(dualGTpat(L)), GTindex(N), t]
                                   for L in basis(λ))^2 / dim(λ)
            end
            K[k + 1, i] = round(K[k + 1, i]; digits = 25)
        end
        target[k + 1] = dim(ν) / D(n, d)
    end
    return K, target
end

## Maximizing the entropy

We maximize $S = -\sum_\nu d_\nu \lVert v_\nu\rVert^2 \log_2 \lVert v_\nu\rVert^2$ over probability
vectors $\{q_\lambda\}$ with `Ipopt`.  The objective is non-convex, so we start from a random
feasible point; in the small ranges considered here the optimum is stable across restarts.

In [ ]:
function maximize_entropy(n, d)
    model = Model(Ipopt.Optimizer)
    set_silent(model)

    l    = length(λirreps(n, d))
    init = rand(l); init ./= sum(init)            # random feasible starting point
    @variable(model, q_λ[i = 1:l] >= 0, start = init[i])
    @constraint(model, sum(q_λ) == 1)

    transition_matrix = LinSystem(n, d)[1]
    v_νs = transition_matrix * q_λ                # the squared norms ‖v_ν‖² (affine in q_λ)
    obj = if d == 2
        sum(-v * log2(v / dim(SUNIrrep(2 * (n % 2 + 2 * (k - 1)), 0))) for (k, v) in enumerate(v_νs))
    else
        sum(-v * log2(v / dim(ν_irrep(k - 1, d))) for (k, v) in enumerate(v_νs))
    end
    @objective(model, Max, obj)

    optimize!(model)

    q_star = Dict(λirreps(n, d)[i] => abs(round(value(q_λ[i]); digits = 6)) for i in 1:l)
    return objective_value(model), transition_matrix, q_star
end

## Example: $n = 2$, $d = 3$

In [ ]:
n, d = 2, 3
f_star, transition_matrix, q_star = maximize_entropy(n, d)

display(q_star)
println("max entropy       = ", f_star)
println("conjectured max   = ", entropy_guess(n, d), "   (= 2·log₂ d_θ)")
println("ratio             = ", f_star / entropy_guess(n, d))

# Compare the optimized block weights with the target distribution that would saturate the bound:
# display([dim(ν_irrep(k, d)) / D(n, d) for k in 0:n])

## Optimized Holevo information vs. the upper bound

The reachable range of $(n, d)$ is small: every entry of `LinSystem` needs many $U(d)$
Clebsch–Gordan coefficients, whose cost grows quickly.

In [ ]:
max_copies = [20, 16, 10, 7, 5, 4, 4, 3, 3, 2]   # largest n attempted, indexed by d = 2,…,11
results_opt = Dict(d => [maximize_entropy(n, d)[1] for n in 1:max_copies[d - 1]] for d in 2:11)

In [ ]:
plt = plot(title = "optimized Holevo information / upper bound", xlabel = "n", legend = :bottomright)
for d in 2:11
    plot!(plt, [ent / log2(binomial(n + 2(d - 1), 2(d - 1))) for (n, ent) in enumerate(results_opt[d])], label = "d = $d")
end
plt

In [ ]:
plt = plot(title = "optimized Holevo information / conjectured maximum", xlabel = "n", legend = :bottomright)
for d in 2:11
    plot!(plt, [ent / entropy_guess(n, d) for (n, ent) in enumerate(results_opt[d])], label = "d = $d")
end
plt

## Probe state supported only on the symmetric subspace

Setting $q_\lambda = \delta_{\lambda,\theta_n}$ corresponds to selecting one column of the
transition matrix (the last column for $d = 2$, the first for $d \ge 3$).  This is the case
studied in closed form — for much larger $n, d$ — in
`symm_subspace_holevo_information_d_gtreq_3.ipynb`.

In [ ]:
function symm_entropy(n, d)
    if d == 2
        v = LinSystem(n, 2)[1][:, end]
        return sum(-val * log2(val / dim(SUNIrrep(2 * (n % 2 + 2 * (k - 1)), 0))) for (k, val) in enumerate(v))
    else
        v = LinSystem(n, d)[1][:, 1]
        return sum(-val * log2(val / dim(ν_irrep(k - 1, d))) for (k, val) in enumerate(v))
    end
end

max_copies_symm = [20, 16, 10, 7, 5, 4, 4, 3, 3, 2]   # largest n, indexed by d = 2,…,11
results_symm = Dict(d => [symm_entropy(n, d) for n in 1:max_copies_symm[d - 1]] for d in 2:11)

In [ ]:
plt = plot(title = "symmetric-subspace Holevo information / upper bound", xlabel = "n", legend = :bottomright)
for d in 2:11
    plot!(plt, [ent / log2(binomial(n + 2(d - 1), 2(d - 1))) for (n, ent) in enumerate(results_symm[d])], label = "d = $d")
end
plt

## Qubit case: comparison with the Yang–Renner–Chiribella probe state

For $d = 2$ the optimal weights solve a small linear system exactly.  We compare them to the
YRC weights $q_j \propto (2j+1)^2$ (i.e. $d_\lambda^2$).

In [ ]:
n = 10
K, target = LinSystemQubits(n)
display(K)
q_j = inv(K) * target
display(q_j)
@show sum(q_j)

# Check that the solution is a valid distribution for every n:
# all(sum(inv(LinSystemQubits(n)[1]) * LinSystemQubits(n)[2]) ≈ 1 for n in 1:60)

In [ ]:
dim_j(n) = [2 * ((n % 2) / 2 + i - 1) + 1 for i in 1:div(n, 2) + 1]   # the values 2j+1
q_yrc(n) = [d^2 / sum(dim_j(n) .^ 2) for d in dim_j(n)]               # YRC weights ∝ d_λ²
q_opt(n) = inv(LinSystemQubits(n)[1]) * LinSystemQubits(n)[2]

n = 65
plot(q_opt(n), label = "optimal \$q_j\$", xlabel = "j index")
plot!(q_yrc(n), label = "YRC \$q_j\$", title = "qubit probe-state weights, n = $n")